# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kuteesatendojeremiah/Tendojerry-Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**My lane is a ranking/scoring task, implemented with a classifier.**

The question my lane answers is not "will this page decline?" in isolation — it is **"which pages should the content team refresh first?"** Editors have a fixed budget (a writer can only rework a handful of pages per cycle), so the useful product is an **ordered queue**: every page gets a priority score, and the team works from the top.

Under the hood, the mechanism is **binary classification**: a model learns to estimate the probability that a page is declining, and that probability becomes the ranking score. So: classification is the *method*, ranking is the *product*. That is why the metric that matters is precision@K (how many of the top-K queue entries are truly worth refreshing) rather than plain accuracy — the code below shows why: with a base rate this high/low, accuracy on all 30,000 pages says nothing about whether the *top 50* picks are good.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

# Bootstrap: clone + install only in a fresh Colab kernel; safe to re-run (no nested clones)
if IN_COLAB and not os.path.exists(DATA_PATH):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

assert os.path.exists(DATA_PATH), "starter CSV not found — are you at the repo root?"

import pandas as pd
df = pd.read_csv(DATA_PATH)

K = 50  # a realistic refresh budget for one cycle
n_pages = len(df)
n_declining = int((df["trend_direction"] == "down").sum())
base_rate = n_declining / n_pages

print(f"Pages in the starter slice:            {n_pages:,}")
print(f"Pages measured as declining ('down'):  {n_declining:,}  ({base_rate:.1%} base rate)")
print(f"Refresh budget per cycle (K):          {K}")
print()
print(f"Expected true decliners in a RANDOM top-{K} pick: ~{base_rate * K:.0f} of {K}")
print(f"-> the task is to ORDER pages so the top {K} beats that number by a wide margin (precision@{K}).")

Pages in the starter slice:            30,000
Pages measured as declining ('down'):  16,262  (54.2% base rate)
Refresh budget per cycle (K):          50

Expected true decliners in a RANDOM top-50 pick: ~27 of 50
-> the task is to ORDER pages so the top 50 beats that number by a wide margin (precision@50).


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**The target is `is_declining_label` — and it is a proxy, not an observed outcome.**

The label is *defined by a rule*: `is_declining_label = (trend_direction == "down")`, and `trend_direction` is itself computed from `trend_pct` (the measured change in the page's trailing-90-day performance). So the model does not learn "this page will lose traffic next month" from a later time window — it learns to recognize pages that the trend rule *already* flags as declining, using only the other signals.

Being honest about this has two consequences:

1. **Leakage rule:** because the label is derived from `trend_direction`/`trend_pct`, those two columns can **never** be features. A model given them would just re-read the answer key and score perfectly while learning nothing.
2. **Careful claims:** what we build is decision support — "pages that look like the measured decliners" — not a causal prediction of future traffic. In weeks 3+, the full warehouse (daily history) makes a true *observed* label possible: define decline in a later window and predict it from an earlier one.

In [2]:
# The label as the pipeline defines it (scripts/01_prepare_features.py):
#   is_declining_label = (trend_direction == "down")
is_declining_label = (df["trend_direction"] == "down").astype(int)

print("Label counts (1 = declining):")
print(is_declining_label.value_counts().rename("pages").to_string())
print()
print("trend_direction values behind the label:")
print(df["trend_direction"].value_counts(dropna=False).to_string())
print()

# Why trend_pct / trend_direction can never be features: they *are* the label.
print("trend_pct range within each label class (deterministic relationship = leakage):")
print(df.groupby(is_declining_label.rename("label"))["trend_pct"].agg(["min", "max", "mean"]).round(1).to_string())

Label counts (1 = declining):
trend_direction
1    16262
0    13738

trend_direction values behind the label:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

trend_pct range within each label class (deterministic relationship = leakage):
         min      max  mean
label                      
0      -20.0  44900.0  79.0
1     -100.0    -20.0 -58.1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**My metric is Precision@50: of the top 50 pages in the ranked queue, what fraction are truly declining?**

Why I can defend it:

- **It matches the decision.** From my week-1 framing: a wrong recommendation costs 4–6 writer-hours. The team only acts on the top of the queue, so quality *at the top* is what matters — K = 50 is a realistic refresh budget for a cycle. Precision on the whole 30,000 rows would reward the model for being right about pages nobody will ever touch.
- **It has honest reference points, named before training.** A random pick scores about the base rate (Section 1). The repo's transparent hand-written rule baseline scores **0.240**. "Good" for me means a clear multiple of the rule baseline — the reference pipeline's random forest lands around ~0.7, i.e. roughly a **3× lift**. The lift is the stable claim; the third decimal is not (it moves with library versions).
- **I can compute it today**, before any model exists — the code below runs it on two zero-ML rankings to set the floor.

In [3]:
import sys
sys.path.insert(0, "scripts")
from ml_utils import precision_at_k  # the repo's canonical metric

import numpy as np

y_true = (df["trend_direction"] == "down").astype(int)
K = 50

# Floor #1: random ordering (what "no signal at all" scores)
rng = np.random.default_rng(42)
p_random = precision_at_k(y_true, rng.random(len(df)), k=K)

# Floor #2: one naive single-signal rule — worst engagement first
# (engagement_rate is a x100 percentage: 5.88 means 5.88%)
p_naive = precision_at_k(y_true, -df["engagement_rate"].fillna(0), k=K)

print(f"Precision@{K}, random ordering:            {p_random:.3f}   (~ base rate)")
print(f"Precision@{K}, naive 'lowest engagement':  {p_naive:.3f}")
print(f"Precision@{K}, repo hand-rule baseline:    0.240   (scripts/02_baseline_score.py)")
print()
print(f"'Good' = a clear multiple of 0.240; the reference random forest reaches ~0.7 (~3x lift).")

Precision@50, random ordering:            0.440   (~ base rate)
Precision@50, naive 'lowest engagement':  0.600
Precision@50, repo hand-rule baseline:    0.240   (scripts/02_baseline_score.py)

'Good' = a clear multiple of 0.240; the reference random forest reaches ~0.7 (~3x lift).


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymized content item (a page) for one client, described by its trailing-90-day search and engagement metrics.**

The starter slice is 30,000 rows × 44 columns across 32 clients. Reading notes for the display below:

- `content_id` / `client_id` are pseudonyms — they identify rows for grouping and joining (e.g. client-level train/test splits) but are **never** model features.
- Rate columns (`ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`) are ×100 percentages: `ctr = 0.76` means 0.76%, not 76%.
- `avg_position = 0` means "no position data", not rank zero.

The grain check below confirms the "one row = one content item" claim instead of assuming it.

In [4]:
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Unique clients:       {df['client_id'].nunique()}")
print(f"Duplicate content_id rows (should be 0): {df['content_id'].duplicated().sum()}")
print()

# One row = one content item: show a readable slice of the columns that matter for my lane
show_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "word_count", "avg_position", "position_tier", "ctr",
    "engagement_rate", "trend_direction",
]
display(df[show_cols].head(5))

Shape: 30,000 rows x 44 columns
Unique content items: 30,000
Unique clients:       32
Duplicate content_id rows (should be 0): 0



,content_id,client_id,content_type,main_intent,word_count,avg_position,position_tier,ctr,engagement_rate,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,10.6,striking,0.76,5.88,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,20.3,page_3_5,0.05,0.00,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,36.5,page_3_5,0.09,0.00,down
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,6.2,page_1,0.49,1.28,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,44.0,page_3_5,0.13,0.00,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Because no single column separates decliners cleanly, and the signals interact.**

Three observations from the data that a hand-written if-statement can't handle:

1. **Every single signal is weak alone.** The medians below barely differ between declining and non-declining pages — and Section 3 already showed what one signal buys as a ranker. A threshold on one column is a coin flip with extra steps.
2. **Signals mean different things in context.** A low CTR is alarming for a page ranking in striking distance but expected for one on page 3–5; missing keyword data follows `content_type`, so even "is this field empty" carries category information. The decline-rate table below shifts across combinations — a surface, not a line.
3. **Rules go stale.** Hand-tuned thresholds are frozen guesses; a model re-fit on fresh data adjusts its weights as the mix of content and clients shifts.

The honest evidence for "ML earns its place" is the measured gap on the same metric: hand-rule baseline **0.240** Precision@50 vs the reference random forest at **~0.7** — about a 3× lift from combining the same signals the rule uses, plus the rest of the feature set.

In [5]:
label = (df["trend_direction"] == "down").astype(int)

# 1) Single signals separate only weakly on their own
candidates = ["avg_position", "ctr", "engagement_rate", "scroll_rate", "word_count", "search_volume"]
print("Median of each candidate signal, declining (1) vs not (0):")
print(df.groupby(label.rename("declining"))[candidates].median().round(2).to_string())

# 2) ...and they interact: the decline rate shifts across category combinations
pivot = df.assign(declining=label).pivot_table(
    index="position_tier", columns="main_intent", values="declining", aggfunc="mean"
).round(2)
print("\nShare of declining pages by position_tier x main_intent:")
print(pivot.to_string())
print("\nNo single-column threshold reproduces this surface — combining many weak,")
print("interacting signals into one ranking score is exactly the job for a model.")

Median of each candidate signal, declining (1) vs not (0):
           avg_position   ctr  engagement_rate  scroll_rate  word_count  search_volume
declining                                                                             
0                 10.05  0.04              0.0         4.21      2839.0           10.0
1                 11.30  0.08              0.0         5.77      2909.0           10.0

Share of declining pages by position_tier x main_intent:
main_intent    commercial  informational  navigational  transactional
position_tier                                                        
deep                 0.32           0.34          0.00           0.36
page_1               0.58           0.59          0.46           0.56
page_3_5             0.54           0.58          0.50           0.53
striking             0.60           0.61          0.33           0.61
top_3                0.39           0.38          0.12           0.37

No single-column threshold reproduces this s

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.